# ML Training Suite - Getting Started

This notebook demonstrates how to use the ML training suite for audio classification.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torchaudio
import matplotlib.pyplot as plt
import numpy as np

# Check device
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'Using device: {device}')

## 1. Load and Visualize Audio

In [ ]:
from src.utils.audio import load_audio, extract_mel_spectrogram, normalize_audio

# Load an audio file (update path to your audio file)
audio_path = '/Volumes/sbdrive/128547_PianoMoodHappy6.wav.mp3'

waveform, sr = load_audio(audio_path, target_sr=22050, duration=5.0)
waveform = normalize_audio(waveform)

print(f'Waveform shape: {waveform.shape}')
print(f'Sample rate: {sr}')
print(f'Duration: {waveform.shape[1] / sr:.2f}s')

In [ ]:
# Plot waveform
plt.figure(figsize=(12, 4))
plt.plot(waveform[0].numpy())
plt.title('Waveform')
plt.xlabel('Sample')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

In [ ]:
# Extract and plot mel spectrogram
mel_spec = extract_mel_spectrogram(waveform, sample_rate=sr)

plt.figure(figsize=(12, 4))
plt.imshow(mel_spec[0].numpy(), aspect='auto', origin='lower', cmap='magma')
plt.colorbar(label='Normalized Amplitude')
plt.title('Mel Spectrogram')
plt.xlabel('Time Frame')
plt.ylabel('Mel Band')
plt.tight_layout()
plt.show()

print(f'Mel spectrogram shape: {mel_spec.shape}')

## 2. Create a Dataset

In [ ]:
from src.data import AudioDataset, create_dataloaders

# Create dataset from directory
# For class-based loading, organize files into subdirectories by class
dataset = AudioDataset.from_directory(
    root_dir='/Volumes/sbdrive',  # Update to your audio directory
    sample_rate=22050,
    duration=5.0,
)

print(f'Dataset size: {len(dataset)}')

# Get a sample
features, label = dataset[0]
print(f'Features shape: {features.shape}')
print(f'Label: {label}')

## 3. Create and Test Model

In [ ]:
from src.models import AudioClassifier

# Create model
model = AudioClassifier(
    num_classes=10,
    n_mels=128,
    channels=[32, 64, 128, 256],
    dropout=0.5,
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

# Test forward pass
x = torch.randn(4, 1, 128, 216)  # batch, channels, mel_bands, time
y = model(x)
print(f'Input shape: {x.shape}')
print(f'Output shape: {y.shape}')

## 4. Training Example

In [ ]:
from src.training import Trainer
import torch.nn as nn

# Create dataloaders (using small batch for demo)
train_loader, val_loader, test_loader = create_dataloaders(
    dataset,
    batch_size=8,
    train_split=0.8,
    val_split=0.1,
    num_workers=0,  # Use 0 for notebook
)

# Setup training
model = AudioClassifier(num_classes=10)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    checkpoint_dir='../models/checkpoints',
    log_dir='../logs',
)

print('Ready to train!')
print('Run: trainer.train(train_loader, val_loader, epochs=10)')

## 5. TensorBoard

Launch TensorBoard to monitor training:

```bash
tensorboard --logdir=logs
```

Then open http://localhost:6006 in your browser.